# Feature Engineering

### Setup

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# Setup working directory
from pathlib import Path
import os

def find_root_dir(marker='tfg'):
    p = Path.cwd()
    for candidate in [p] + list(p.parents):
        if candidate.name == marker:
            return candidate.resolve()
        if (candidate / marker).is_dir():
            return (candidate / marker).resolve()
    raise FileNotFoundError(f"Could not find '{marker}' folder in {Path.cwd()} or its parents.")

os.chdir(find_root_dir('tfg'))

## Load Data

In [3]:
REFERENCE_DATASET_PATH = "data/processed/stage1/reference-8k-baseline.parquet"

In [4]:
from src.preprocessing import load_dataset

df = load_dataset(REFERENCE_DATASET_PATH)
print("\nFirst few rows:")
df.head()


First few rows:


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,172.16.0.1-192.168.10.50-59601-80-6,172.16.0.1,59601,192.168.10.50,80,6,7/7/2017 4:02,1148952,5,0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DDoS
1,172.16.0.1-192.168.10.50-28809-80-6,172.16.0.1,28809,192.168.10.50,80,6,7/7/2017 4:07,8361212,4,0,...,20,581.0,0.0,581,581,8360631.0,0.0,8360631,8360631,DDoS
2,172.16.0.1-192.168.10.50-49759-80-6,172.16.0.1,49759,192.168.10.50,80,6,7/7/2017 3:56,1280276,2,5,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DDoS
3,172.16.0.1-192.168.10.50-60971-80-6,172.16.0.1,60971,192.168.10.50,80,6,7/7/2017 4:12,78218,3,6,...,20,0.0,0.0,0,0,0.0,0.0,0,0,DDoS
4,192.168.10.3-192.168.10.9-53-51832-17,192.168.10.9,51832,192.168.10.3,53,17,7/7/2017 4:16,223,2,2,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


## Baseline Dataset
- Numeric/statistical features only

In [5]:
CATEGORICAL_COLUMNS = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Source Port",
    "Destination Port",
    "Protocol",
    "Timestamp"
]
df_baseline = df.drop(columns=CATEGORICAL_COLUMNS)
df_baseline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7997 entries, 0 to 7996
Data columns (total 68 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Flow Duration                7997 non-null   int32  
 1   Total Fwd Packets            7997 non-null   int16  
 2   Total Backward Packets       7997 non-null   int16  
 3   Total Length of Fwd Packets  7997 non-null   int32  
 4   Total Length of Bwd Packets  7997 non-null   int32  
 5   Fwd Packet Length Max        7997 non-null   int16  
 6   Fwd Packet Length Min        7997 non-null   int16  
 7   Fwd Packet Length Mean       7997 non-null   float32
 8   Fwd Packet Length Std        7997 non-null   float32
 9   Bwd Packet Length Max        7997 non-null   int16  
 10  Bwd Packet Length Min        7997 non-null   int16  
 11  Bwd Packet Length Mean       7997 non-null   float32
 12  Bwd Packet Length Std        7997 non-null   float32
 13  Flow Bytes/s      

## Encoded Dataset
- Numeric/statistical features and encoded categorical features

In [6]:
CATEGORICAL_COLUMNS = [
    "Flow ID",
    "Timestamp"
]
df_encoded = df.drop(columns=CATEGORICAL_COLUMNS)

In [7]:
from src.feature_engineering import encode_categorical_features
df_encoded = encode_categorical_features(df_encoded)

Encoding Categorical Features
- Encoding Source/Destination IP
- Encoding Protocol
- Encoding Source/Destination Port
- Sample of scaled '{Source, Destination} IP':
      Source IP  Destination IP
5552   0.865553        0.871350
4611   0.770547        0.871350
1860   0.865553        0.059087
- Sample of new protocol column:
      is_0  is_6  is_17
1764   0.0   1.0    0.0
4069   0.0   1.0    0.0
6324   0.0   1.0    0.0
- Sample of encoded '{Source, Destination} Port':
      Source Port_Dynamic  Source Port_Registered  Source Port_Well-known  \
2748                  0.0                     0.0                     1.0   
3287                  1.0                     0.0                     0.0   
6079                  1.0                     0.0                     0.0   

      Destination Port_Dynamic  Destination Port_Registered  \
2748                       0.0                          1.0   
3287                       0.0                          0.0   
6079                       0.0

In [8]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7997 entries, 0 to 7996
Data columns (total 79 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Source IP                    7997 non-null   float64
 1   Destination IP               7997 non-null   float64
 2   Flow Duration                7997 non-null   int32  
 3   Total Fwd Packets            7997 non-null   int16  
 4   Total Backward Packets       7997 non-null   int16  
 5   Total Length of Fwd Packets  7997 non-null   int32  
 6   Total Length of Bwd Packets  7997 non-null   int32  
 7   Fwd Packet Length Max        7997 non-null   int16  
 8   Fwd Packet Length Min        7997 non-null   int16  
 9   Fwd Packet Length Mean       7997 non-null   float32
 10  Fwd Packet Length Std        7997 non-null   float32
 11  Bwd Packet Length Max        7997 non-null   int16  
 12  Bwd Packet Length Min        7997 non-null   int16  
 13  Bwd Packet Length 

## Save Datasets

In [9]:
from src.preprocessing import save
save("stage1/reference-8k-numeric.parquet", df_baseline)
save("stage1/reference-8k-encoded.parquet", df_encoded)

Saving dataset as .parquet
Final Dataset Saved: data/processed\stage1/reference-8k-numeric.parquet (1.04 MB) 
Saving dataset as .parquet
Final Dataset Saved: data/processed\stage1/reference-8k-encoded.parquet (1.07 MB) 
